In [ ]:
import pandas as pd

url = "https://github.com/masterfloss/data/raw/refs/heads/main/fake_new.xlsx"
df = pd.read_excel(url)

# Basic cleaning
df = df.dropna()

display(df.columns)
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_text = df["text"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

mlp_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(1, activation='sigmoid')
])

mlp_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

mlp_model.fit(X_train, y_train, epochs=5, batch_size=32)


In [ ]:
loss, accuracy = mlp_model.evaluate(X_test, y_test)
print("MLP Accuracy:", accuracy)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)


In [ ]:
lstm_model = keras.Sequential([
    layers.Embedding(5000, 64, input_length=100),
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

lstm_model.fit(X_train_pad, y_train, epochs=5, batch_size=32)


In [ ]:
loss, accuracy = lstm_model.evaluate(X_test_pad, y_test)
print("LSTM Accuracy:", accuracy)


In [ ]:
cnn_model = keras.Sequential([
    layers.Embedding(5000, 64, input_length=100),
    layers.Conv1D(64, 5, activation='relu'),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

cnn_model.fit(X_train_pad, y_train, epochs=5, batch_size=32)


In [ ]:
loss, accuracy = cnn_model.evaluate(X_test_pad, y_test)
print("CNN Accuracy:", accuracy)


In [ ]:
from sklearn.metrics import accuracy_score

# Example with MLP
y_pred = (mlp_model.predict(X_test) > 0.5).astype("int32")
print("Manual Accuracy:", accuracy_score(y_test, y_pred))


In [ ]:
results = {
    "MLP": mlp_model.evaluate(X_test, y_test)[1],
    "LSTM": lstm_model.evaluate(X_test_pad, y_test)[1],
    "CNN": cnn_model.evaluate(X_test_pad, y_test)[1]
}

print(results)
